<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания: 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс PaymentMethod в C#, который будет представлять
различные способы оплаты. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.
Требования к базовому классу PaymentMethod:
• Атрибуты: ID способа оплаты (PaymentMethodId), Название способа оплаты
(MethodName), Минимальная сумма (MinAmount).
• Методы:
o ProcessPayment(decimal amount): метод для обработки платежа
указанной суммы.
o CheckMinimumAmount(decimal amount): метод для проверки
минимальной суммы платежа.
o GetPaymentDetails(): метод для получения деталей способа оплаты.

Требования к производным классам:
1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные
атрибуты, такие как URL платежной системы (PaymentUrl).
Метод ProcessPayment() должен быть переопределен для включения URL
платежной системы в процесс оплаты.
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные
атрибуты, такие как Банковские данные (BankData).
Метод CheckMinimumAmount() должен быть переопределен для проверки
минимальной суммы платежа с учетом банковских комиссий.
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать
дополнительные атрибуты, такие как Место выдачи наличных
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен
для отображения места выдачи наличных.


----


#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
public class PaymentMethod
{
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;

    public int PaymentMethodId
    {
        get { return _paymentMethodId; }
        set
        {
            if (value <= 0)
                throw new ArgumentException("ID должен быть больше 0.");
            _paymentMethodId = value;
        }
    }
    public string MethodName
    {
        get { return _methodName; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Название не может быть пустым.");
            _methodName = value;
        }
    }
    public decimal MinAmount
    {
        get { return _minAmount; }
        set
        {
            if (value < 0)
                throw new ArgumentException("Минимальная сумма не может быть отрицательной.");
            _minAmount = value;
        }
    }

    public PaymentMethod(int paymentMethodId, string methodName, decimal minAmount)
    {
        PaymentMethodId = paymentMethodId;
        MethodName = methodName;
        MinAmount = minAmount;
    }

    public virtual bool ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
            return false;

        Console.WriteLine($"Платёж на сумму {amount} через {MethodName}.");
        return true;
    }
    public virtual bool CheckMinimumAmount(decimal amount)
    {
        bool isValid = amount >= MinAmount;
        if (isValid)
        {
            Console.WriteLine($"Сумма {amount} соответствует минимальной сумме {MinAmount}.");
        }
        else
        {
            Console.WriteLine($"Сумма {amount} меньше минимальной суммы {MinAmount}.");
        }
        return isValid;
    }
    public virtual void GetPaymentDetails()
    {
        Console.WriteLine($"ID: {PaymentMethodId}, Название: {MethodName}, Минимальная сумма: {MinAmount}");
    }
    public virtual decimal CalculateFee(decimal amount) => 0m;
}

public class OnlinePayment : PaymentMethod
{
    private string _paymentUrl;

    public string PaymentUrl
    {
        get { return _paymentUrl; }
        set
        {
            if (string.IsNullOrWhiteSpace(value) || !value.StartsWith("http"))
                throw new ArgumentException("URL должен начинаться с http.");
            _paymentUrl = value;
        }
    }
    public OnlinePayment(int paymentMethodId, string methodName, decimal minAmount, string paymentUrl)
        : base(paymentMethodId, methodName, minAmount)
    {
        PaymentUrl = paymentUrl;
    }

    public override bool ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
            return false;

        Console.WriteLine($"Онлайн оплата на сумму {amount} по ссылке {PaymentUrl}.");
        return true;
    }
}

public class BankTransfer : PaymentMethod
{
    private string _bankData;
    private const decimal TransferFee = 0.1m;

    public string BankData
    {
        get { return _bankData; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Банковские данные не могут быть пустыми.");
            _bankData = value;
        }
    }

    public BankTransfer(int paymentMethodId, string methodName, decimal minAmount, string bankData)
        : base(paymentMethodId, methodName, minAmount)
    {
        BankData = bankData;
    }
    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal amountAfterFee = amount - amount * TransferFee;
        Console.WriteLine($"Комиссия {TransferFee * 100}%, сумма после комиссии: {amountAfterFee}");
        return base.CheckMinimumAmount(amountAfterFee);
    }

    public override decimal CalculateFee(decimal amount) => amount * TransferFee;

    public override bool ProcessPayment(decimal amount)
    {
        if (!CheckMinimumAmount(amount))
            return false;

        decimal fee = CalculateFee(amount);
        Console.WriteLine($"Банковский перевод на сумму {amount}. Комиссия: {fee} ({TransferFee * 100}%), получатель получит {amount - fee}.");
        return true;
    }

    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Банковские данные: {BankData}, комиссия: {TransferFee * 100}%");
    }
}

public class CashPayment : PaymentMethod
{
    private string _cashPickupPoint;

    public string CashPickupPoint
    {
        get { return _cashPickupPoint; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Место выдачи не может быть пустым.");
            _cashPickupPoint = value;
        }
    }
    public CashPayment(int paymentMethodId, string methodName, decimal minAmount, string cashPickupPoint)
        : base(paymentMethodId, methodName, minAmount)
    {
        CashPickupPoint = cashPickupPoint;
    }

    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Место получения наличных: {CashPickupPoint}");
    }
}




The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
public class Shop
{
    private string _name;
    private decimal _revenue;

    public string Name
    {
        get { return _name; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Название магазина не может быть пустым.");
            _name = value;
        }
    }

    public decimal Revenue
    {
        get { return _revenue; }
        private set { _revenue = value; }
    }

    public Shop(string name)
    {
        Name = name;
    }

    public void Receive(decimal amount)
    {
        Revenue += amount;
        Console.WriteLine($"Магазин {Name} получил {amount}. Выручка: {Revenue}");
    }
}


public class Customer
{
    private string _name;
    private decimal _balance;
    private List<PaymentMethod> _paymentMethods = new List<PaymentMethod>();

    public string Name
    {
        get { return _name; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Имя не может быть пустым.");
            _name = value;
        }
    }

    public decimal Balance
    {
        get { return _balance; }
        private set
        {
            if (value < 0)
                throw new ArgumentException("Баланс не может быть отрицательным.");
            _balance = value;
        }
    }

    public Customer(string name, decimal balance)
    {
        Name = name;
        Balance = balance;
    }

    public void AddPaymentMethod(PaymentMethod method)
    {
        if (method == null)
            throw new ArgumentNullException(nameof(method));
        _paymentMethods.Add(method);
        Console.WriteLine($"{Name} добавил способ оплаты: {method.MethodName}");
    }

    public bool Pay(Shop shop, PaymentMethod method, decimal amount)
    {
        Console.WriteLine($"\n--- {Name} платит {amount} в магазине {shop.Name} ({method.MethodName}) ---");

        if (!_paymentMethods.Contains(method))
        {
            Console.WriteLine("Этот способ оплаты не подключён у клиента.");
            return false;
        }

        if (amount > Balance)
        {
            Console.WriteLine($"Недостаточно средств. Баланс: {Balance}");
            return false;
        }

        if (!method.ProcessPayment(amount))
        {
            Console.WriteLine("Платёж отклонён.");
            return false;
        }

        decimal fee = method.CalculateFee(amount);

        Balance -= amount;
        shop.Receive(amount - fee);
        Console.WriteLine($"Платёж прошёл. Баланс {Name}: {Balance}");
        return true;
    }
}

In [ ]:
var online = new OnlinePayment(1, "Онлайн оплата", 10m, "https://pay.example.com");
var bank = new BankTransfer(2, "Банковский перевод", 50m, "Данные");
var cash = new CashPayment(3, "Наличные", 5m, "Пункт выдачи №7");

var customer = new Customer("Иван", 500m);
var shop = new Shop("TechStore");

customer.AddPaymentMethod(online);
customer.AddPaymentMethod(bank);

PaymentMethod[] methods = { online, bank, cash };
foreach (var m in methods)
{
    m.GetPaymentDetails();
    Console.WriteLine();
}

customer.Pay(shop, online, 100m);   
customer.Pay(shop, bank, 55m);      
customer.Pay(shop, bank, 100m);     
customer.Pay(shop, cash, 20m);      
customer.Pay(shop, online, 1000m);  

try { online.PaymentUrl = "ftp://bad"; }
catch (ArgumentException e) { Console.WriteLine($"\nОшибка: {e.Message}"); }

Console.WriteLine($"\nИтог: баланс {customer.Name} = {customer.Balance}, выручка {shop.Name} = {shop.Revenue}");

Иван добавил способ оплаты: Онлайн оплата
Иван добавил способ оплаты: Банковский перевод
ID: 1, Название: Онлайн оплата, Минимальная сумма: 10

ID: 2, Название: Банковский перевод, Минимальная сумма: 50
Банковские данные: IBAN PL00 0000 0000, комиссия: 10,0%

ID: 3, Название: Наличные, Минимальная сумма: 5
Место получения наличных: Пункт выдачи №7


--- Иван платит 100 в магазине TechStore (Онлайн оплата) ---
Сумма 100 соответствует минимальной сумме 10.
Онлайн оплата на сумму 100 по ссылке https://pay.example.com.
Магазин TechStore получил 100. Выручка: 100
Платёж прошёл. Баланс Иван: 400

--- Иван платит 55 в магазине TechStore (Банковский перевод) ---
Комиссия 10,0%, сумма после комиссии: 49,5
Сумма 49,5 меньше минимальной суммы 50.
Платёж отклонён.

--- Иван платит 100 в магазине TechStore (Банковский перевод) ---
Комиссия 10,0%, сумма после комиссии: 90,0
Сумма 90,0 соответствует минимальной сумме 50.
Банковский перевод на сумму 100. Комиссия: 10,0 (10,0%), получатель получит 90,0